# DBSCAN — Density-Based Spatial Clustering

**Course**: CMOR 438 / INDE 577 — Data Science & Machine Learning  
**Dataset**: International Football Results (1872–present)  
**Author**: Neriah29  

---

## What DBSCAN Fixes

K-Means has two fundamental limitations:
1. You must specify K upfront
2. It assumes clusters are roughly circular (Euclidean distance to centroid)

DBSCAN fixes both. It finds **dense regions** of any shape, automatically determines how many clusters exist, and explicitly marks outliers as noise.

---

## The Core Idea

> *A cluster is a dense region of points separated from other dense regions by sparse space.*

Two parameters define "dense":
- **eps (ε)**: the neighborhood radius — how far to look around each point
- **min_samples**: how many points must be within eps to call a region dense

### Three types of points

| Type | Definition | Role |
|---|---|---|
| **Core point** | ≥ min_samples neighbors within eps | Defines the cluster interior |
| **Border point** | < min_samples neighbors, but within eps of a core point | Cluster edge |
| **Noise point** | Neither core nor border | Labeled -1, belongs to no cluster |

### Cluster expansion

Starting from an unvisited core point, DBSCAN expands outward — adding all reachable points to the cluster. It keeps expanding through any core points it encounters. When no more core points can be reached, the cluster is complete. Repeat for the next unvisited core point.

---

## DBSCAN vs K-Means

| | K-Means | DBSCAN |
|---|---|---|
| Number of clusters | You specify K | Discovered automatically |
| Cluster shape | Circular | Any shape |
| Noise handling | Every point assigned | Noise → label -1 |
| Parameters | K | eps, min_samples |
| Struggles with | Irregular shapes | Varying density |


---
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons, make_circles

import sys
sys.path.insert(0, '../../src')
from football_ml.unsupervised_learning.dbscan import DBSCAN
from football_ml.unsupervised_learning.kmeans import KMeans

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

---
## 1. Why Shape Matters — DBSCAN vs K-Means

In [ ]:
# Two datasets K-Means handles poorly
X_moons,   _ = make_moons(n_samples=200,  noise=0.08, random_state=SEED)
X_circles, _ = make_circles(n_samples=200, noise=0.05, factor=0.4, random_state=SEED)

datasets = [
    (X_moons,   'Moons',   KMeans(k=2, n_init=5, random_state=SEED),
                           DBSCAN(eps=0.25, min_samples=5)),
    (X_circles, 'Circles', KMeans(k=2, n_init=5, random_state=SEED),
                           DBSCAN(eps=0.3,  min_samples=5)),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

for row, (X, name, km, db) in enumerate(datasets):
    km.fit(X)
    db.fit(X)

    for col, (labels, title) in enumerate([
        (km.labels_,                    f'K-Means (K=2) — {name}'),
        (db.labels_,                    f'DBSCAN — {name} ({db.n_clusters_} clusters, {db.n_noise_} noise)'),
    ]):
        ax = axes[row][col]
        unique = np.unique(labels)
        colors = plt.cm.tab10(np.linspace(0, 0.5, len(unique)))
        for label, color in zip(unique, colors):
            mask = labels == label
            marker = 'x' if label == -1 else 'o'
            ax.scatter(X[mask, 0], X[mask, 1], color=color if label != -1 else 'black',
                       s=20, marker=marker, alpha=0.8,
                       label='Noise' if label == -1 else f'Cluster {label}')
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.legend(fontsize=7)

plt.suptitle('DBSCAN vs K-Means on Non-Spherical Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

K-Means forces circular boundaries — it splits both datasets incorrectly. DBSCAN follows the density and correctly identifies the crescent and ring shapes.

---
## 2. Load & Engineer Features

In [ ]:
df = pd.read_csv('../../data/results.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df['goal_diff']   = df['home_score'] - df['away_score']
df['total_goals'] = df['home_score'] + df['away_score']
df['home_win']    = (df['home_score'] > df['away_score']).astype(int)

def compute_team_rolling_stats(df, window=10):
    home_log = df[['date', 'home_team', 'home_score', 'away_score']].copy()
    home_log.columns = ['date', 'team', 'scored', 'conceded']
    away_log = df[['date', 'away_team', 'away_score', 'home_score']].copy()
    away_log.columns = ['date', 'team', 'scored', 'conceded']
    team_log = pd.concat([home_log, away_log]).sort_values('date').reset_index(drop=True)
    team_log['rolling_scored'] = (
        team_log.groupby('team')['scored']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    team_log['rolling_conceded'] = (
        team_log.groupby('team')['conceded']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    return team_log.drop_duplicates(subset=['date', 'team'], keep='last').set_index(['date', 'team'])

team_stats = compute_team_rolling_stats(df)

def get_stat(row, team_col, stat_col):
    try:
        return team_stats.loc[(row['date'], row[team_col]), stat_col]
    except KeyError:
        return np.nan

df['home_goals_rolling']    = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_scored'), axis=1)
df['home_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_conceded'), axis=1)
df['away_goals_rolling']    = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_scored'), axis=1)
df['away_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_conceded'), axis=1)

home_wins = df.groupby('home_team').apply(lambda g: (g['home_score'] > g['away_score']).mean()).rename('home_win_rate')
away_wins = df.groupby('away_team').apply(lambda g: (g['away_score'] > g['home_score']).mean()).rename('away_win_rate')
df = df.join(home_wins, on='home_team').join(away_wins, on='away_team')
df['neutral'] = df['neutral'].astype(int)

feature_cols = [
    'home_goals_rolling', 'away_goals_rolling',
    'home_conceded_rolling', 'away_conceded_rolling',
    'home_win_rate', 'away_win_rate', 'neutral'
]
df_clean = df[feature_cols + ['goal_diff', 'home_win', 'total_goals']].dropna()
print(f'Dataset: {df_clean.shape[0]} matches')

---
## 3. Prepare Data — Small Sample for Speed

In [ ]:
# DBSCAN computes all pairwise distances — O(n²) memory and time
# Use a sample for the analysis
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(df_clean), size=2000, replace=False)
df_sample  = df_clean.iloc[sample_idx].copy()

X = df_sample[feature_cols].values
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

print(f'Using {len(X_sc)} samples')

---
## 4. Choosing eps — K-Distance Plot

In [ ]:
# For each point, find distance to its k-th nearest neighbor
# Plotting sorted k-distances helps identify a good eps:
# the 'elbow' in the curve is where density drops off sharply

min_samples = 5
from sklearn.neighbors import NearestNeighbors
nn = NearestNeighbors(n_neighbors=min_samples).fit(X_sc)
distances, _ = nn.kneighbors(X_sc)
k_distances = np.sort(distances[:, -1])[::-1]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_distances, color='#4878CF', linewidth=2)
ax.set_xlabel('Points (sorted by distance)', fontsize=12)
ax.set_ylabel(f'{min_samples}-NN Distance', fontsize=12)
ax.set_title('K-Distance Plot — Find the Elbow for eps', fontsize=13, fontweight='bold')
ax.axhline(0.8, color='#E87272', linestyle='--', linewidth=1.5, label='eps candidate')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

The elbow of this curve — where it bends sharply — suggests a good eps value. Points to the left of the elbow are in dense regions; points to the right are in sparse regions (potential noise).

---
## 5. Run DBSCAN

In [ ]:
model = DBSCAN(eps=0.8, min_samples=5).fit(X_sc)

print(f'Clusters found:  {model.n_clusters_}')
print(f'Noise points:    {model.n_noise_} ({model.n_noise_/len(X_sc):.1%} of data)')
print(f'Core points:     {len(model.core_sample_indices_)}')

if model.n_clusters_ >= 2:
    print(f'Silhouette:      {model.silhouette_score(X_sc):.3f}')

# Cluster size distribution
unique, counts = np.unique(model.labels_[model.labels_ != -1], return_counts=True)
for c, n in zip(unique, counts):
    print(f'  Cluster {c}: {n} points')

---
## 6. Cluster Profiles

In [ ]:
df_sample = df_sample.copy()
df_sample['cluster'] = model.labels_

# Separate noise from clusters
df_clusters = df_sample[df_sample['cluster'] != -1]
df_noise    = df_sample[df_sample['cluster'] == -1]

if len(df_clusters['cluster'].unique()) > 0:
    profile = df_clusters.groupby('cluster')[['goal_diff', 'home_win', 'total_goals']].mean()
    print('Cluster profiles:')
    print(profile.round(3).to_string())
    print(f'\nNoise matches: {len(df_noise)} — these are outlier/unusual matches')

---
## 7. Effect of eps and min_samples

In [ ]:
eps_values      = [0.3, 0.5, 0.8, 1.2]
min_samples_values = [3, 5, 10]

print(f'{"eps":>6} {"min_samples":>12} {"clusters":>10} {"noise %":>10}')
print('-' * 42)
for eps in eps_values:
    for ms in min_samples_values:
        m = DBSCAN(eps=eps, min_samples=ms).fit(X_sc)
        print(f'{eps:>6.1f} {ms:>12d} {m.n_clusters_:>10d} {m.n_noise_/len(X_sc):>9.1%}')

### Reading this table

- **Larger eps**: more points within each neighborhood → larger clusters → fewer clusters → less noise
- **Larger min_samples**: harder to be a core point → more noise → fewer, tighter clusters
- The right combination depends on what structure you're looking for in the data

---
## 8. Discussion — Does DBSCAN Suit Football?

### What it does well
- **No K needed**: discovers the number of clusters automatically
- **Finds outliers**: noise points (-1) represent unusual matches — potential anomalies, mismatches, or extraordinary results
- **Arbitrary shapes**: follows the density structure of the data, not just circular regions
- **Robust to outliers**: unlike K-Means, extreme matches don't distort cluster shapes

### What it struggles with
1. **O(n²) complexity**: pairwise distance computation is slow for large datasets — we sampled 2,000 matches
2. **Parameter sensitivity**: eps and min_samples need careful tuning — the k-distance plot helps but doesn't give a definitive answer
3. **Varying density**: if some clusters are tight and others are spread out, no single eps works well for all
4. **Football data is dense and continuous**: with 7 continuous features, there may not be clear density gaps between natural groups

### What the noise points mean for football

Noise points are matches that don't fit any dense region — they're isolated in feature space. In football terms, these might be unusual scorelines, matches between teams with no historical pattern, or genuinely unpredictable encounters. This is actually useful information — a model might want to treat these differently.

---
## Summary

| | |
|---|---|
| **Algorithm type** | Density-based clustering |
| **K required** | No — discovered automatically |
| **Cluster shape** | Any — follows density |
| **Noise handling** | Explicit — labeled -1 |
| **Parameters** | eps (neighborhood radius), min_samples (density threshold) |
| **Complexity** | O(n²) — slow on large datasets |
| **Key advantage over K-Means** | Shape flexibility + noise detection |
| **Key limitation** | Parameter sensitivity, varying density |
